In [1]:
import sys, time, tempfile
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'C'/'evoca.h').exists())
sys.path.insert(0, str(ROOT/'python'))
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import bifurcation as bif
import evoca_explore as ee
from evoca_py import EvoCA
from evoca_nb import rgb, export_recipe_once   # shared notebook helpers: python/evoca_nb.py
WS = 'Criticality-bifurcation'
T0 = time.time()

# Criticality-bifurcation: lab notebook

**Question.** Where are EvoCA's dynamical transitions as control parameters vary (bifurcation diagrams of global time series such as population and activity), and what characterizes them?

Framing (N's established positions): ruggedness is emergent in EvoCA (no NK-style knob); critical exponents are not only a finite-size-scaling question, so a bifurcation-diagram program over control parameters is preferred; Feigenbaum period-doubling has **not** been observed in EvoCA and is not presumed (the `feigenbaum_delta` helper exists but we say "bifurcation" generally).

Links: [`digests/`](digests/) (none yet) and [`Scans/`](Scans/) (none yet; this workstream owns no scans until the first torque sweep). Source of the viability bracket used below: [`../Resource-driving/Scans/2026-05-18_R1_mscale_bracket/notes.md`](../Resource-driving/Scans/2026-05-18_R1_mscale_bracket/notes.md).

**Running the examples interactively.** After each example run there is a code cell of commented-out lines that open the same configuration in the interactive SDL window (`run_with_controls`, from `python/controls.py`). Uncomment and run it to watch the simulation; it is commented out so that *Run All* never blocks on a window. Only one EvoCA lattice exists at a time, so this replaces any simulation still alive in the kernel. Runs now use the capped mouthful (`food_cap=1`, the A9 fix); the outputs stored in this notebook were computed before it.

Snapshot of `ISSUES.md` as of the last execution of this notebook. `ISSUES.md` is the source; edit it there, not here.

In [2]:
display(Markdown((ROOT/'Research'/WS/'ISSUES.md').read_text()))

# Criticality-bifurcation: issues and directions
_Updated 2026-10-15_

## Decisions needed (N)
- **CRIT-D1**: First control parameter, range and N for the torque campaign — options: (a) `food_inc` 0.010-0.06 fine grid (the free-energy source; Phi is its natural partner; the first example diagram already shows extinction, oscillation, amplitude collapse and saturation inside this range), (b) `m_scale` 1.5-6 (R1: interior productive optimum ~2.5-3.5, U-shaped viability), (c) both as a 2-D map. Recommend (a) first at fixed R1-informed values (m_scale 3.5, tax 0.035, mu_lut 0.06, gdiff 0.06), then (b) at food_inc 0.018. Blocks CRIT-N1.
- **CRIT-D2**: What counts as a transition (a pre-registered criterion, so a "flat diagram" is a valid outcome) — recommend all of: extinction boundary (fraction of replicate seeds extinct crosses 0.5), onset of sustained oscillation (spectral peak above a noise floor, not the raw extrema count), amplitude collapse (tail std/mean falling through a threshold), and, for any candidate, an N-scaling test. Blocks CRIT-N1/N4.

## Open questions
- **CRIT-Q1**: Is the observed boom-bust oscillation a finite-size synchronized-lattice cycle (relative amplitude decays with N) or intrinsic? The single-seed example check (notebook) suggests the large-amplitude cycle at food_inc 0.016 persists with N (std/N^2 0.335, 0.318, 0.297 at N=32/64/128) while the small-amplitude regime at 0.022 shrinks (relative amplitude 0.033 -> 0.009 from N=64 to 128); neither is replicated. Decides whether diagram columns mean anything in the thermodynamic limit; couples to the Docs/EvoCA_research_directions.md §5 H_critical / H_ceiling question (xi ~ N/3 plateau).
- **CRIT-Q2**: Is `recurrent_extrema` the right reducer for noisy, irregular-amplitude series? In the example runs the extrema count is a filled band (tens to hundreds of points), so it cannot distinguish limit cycle from irregular dynamics. Candidates: autocorrelation/spectral peak, return map of successive maxima, Poincare-section sampling, largest-Lyapunov-style divergence of nearby seeds. See `python/bifurcation.py` `recurrent_extrema`.
- **CRIT-Q3**: Which side of any boundary is a bifurcation in the dynamical sense versus a stochastic extinction (absorbing-state) transition? EvoCA has an absorbing state (pop=0), so the extinction edge may belong to the directed-percolation-like class rather than a deterministic bifurcation; the R1 extinction table (m_scale x food_inc) is endpoint data only.
- **CRIT-Q4**: Hysteresis: does ramping `food_inc` up versus down give different columns (the directions doc, R3 (c), suggests a slow ramp as an alternative to a fixed grid)? Untested; needs a ramp driver (CRIT-N3).
- **CRIT-Q5**: Does evolution matter? Runs with `mu_lut=0` freeze the substrate (see memory notes on frozen rich substrates); compare mu_lut=0 against mu_lut>0 columns to separate ecological (food-limited) oscillation from evolutionary change.

## Next directions
- **CRIT-N1**: Real `food_inc` bifurcation campaign — campaign (torque), N in {64, 128, 256}, 8000+ ticks, 41+ values x >=5 seeds, observable `pop` first; blocked on CRIT-D1/D2. Uses `python/bifurcation.py` `sweep` unchanged.
- **CRIT-N2**: Multi-observable harness — needs dev (small, Python only): extend `scalar_series` to record Phi (`get_dissipation`), births/deaths, `dyn_activity` total, `A_new` (needs neutral shadow, computed as in `evoca_explore.run_sim`), `lut_div`, lineage return; the example notebook has a working per-tick `record()` prototype. Cross-ref checkpoint actions G1 (A_new), L1 (Phi), D2/D3 (lineage return).
- **CRIT-N3**: Slow-ramp / hysteresis driver — needs dev (Python): change `food_inc` (or `m_scale`) during a run through the live sim attribute instead of re-initializing, up and down.
- **CRIT-N4**: Control-parameter and finite-size follow-up on whichever transition passes CRIT-D2 — analysis/campaign: order parameter = alive density or oscillation amplitude, susceptibility, Binder cumulant across N; fixes the Q1 H_critical / H_ceiling question (directions doc §5, R3 (a)/(b)).
- **CRIT-N5**: Second axes — campaign: `m_scale` (R1 optimum), `tax`, `mu_lut`, then a 2-D map (m_scale x food_inc) where the R1 non-monotone viability suggests structure; start from R1 `results.csv` to pre-select regions.
- **CRIT-N6**: Replace the reducer's raw extrema count by a classifier (fixed point / periodic / irregular / extinct / saturated) for sweep summaries — needs dev; feeds CRIT-Q2. Keep `feigenbaum_delta` but do not report it unless a clean cascade appears.

## Parked
- Feigenbaum delta extraction: period-doubling has not been observed in EvoCA (N's position); helper retained, not used.
- Motility-related parameters: motility branch on hold (memory, 2026-07-19).


## 2026-05-16: harness built (S2e)

**Question.** Can EvoCA's global time series be reduced to a bifurcation diagram without touching the C core?

**What was done.** Agent S2e (commit `dfdb010`, archived in [`../research_board_archival.md`](../research_board_archival.md)) added `python/bifurcation.py`: `scalar_series` (per-tick observable, same init sequence as `run_sim`), `recurrent_extrema` (post-transient local extrema, clustered), `sweep`, `to_scatter`, plus a best-effort `feigenbaum_delta` / `doubling_thresholds`. `tests/test_bifurcation.py` checks the reducer on synthetic fixed-point / period-2 / period-4 series. The `--demo` (food_inc 0.02-0.20, N=48, 400 steps) was a format check only; **no scientific sweep was ever run** (board: "no sweeps launched yet").

**Directions that came out of it.** Docs/EvoCA_research_directions.md §5 plus the NP reply R3 split the exponent questions into three programs: (c) bifurcations of global dynamics (preferred, cheap), (b) control-parameter exponents at fixed N, (a) finite-size scaling only as the xi ~ N/3 plateau discriminator (Binder cumulant / susceptibility). Priority stated there: (c) > (b) > (a).

## 2026-05-18: R1 viability bracket (input from Resource-driving)

R1 (`Resource-driving/Scans/2026-05-18_R1_mscale_bracket`, N=256, 8000 ticks, 300 configs) bracketed `m_scale` and `food_inc`. Recorded finding: the `m_scale` productive optimum is interior (~2.5-3.5); viability is U-shaped in `m_scale` under food scarcity (worst extinction at intermediate `m_scale`, lowest at 5.0); the `food_inc` optimum is at the high end 0.013-0.018 and `food_inc` <= 0.010 is lethal at m_scale 2.5/3.5. These are *endpoint* statistics (extinct yes/no, means), not time-series dynamics, so they locate where to look but say nothing about oscillation vs fixed point. Let us re-read the extinction table from `results.csv` to choose the control parameter.

In [3]:
r1 = pd.read_csv(ROOT/'Research/Resource-driving/Scans/2026-05-18_R1_mscale_bracket/results.csv')
print('extinct fraction by (m_scale, food_inc):')
display(r1.groupby(['param_m_scale','param_food_inc']).extinct.mean().unstack().round(2))
print('R1 held tax in', sorted(r1.param_tax.unique()), ' mu_lut in', sorted(r1.param_mu_lut.unique()), ' gdiff in', sorted(r1.param_gdiff.unique()))

extinct fraction by (m_scale, food_inc):


param_food_inc,0.005,0.008,0.010,0.013,0.018
param_m_scale,,,,,
2.5,1.00,0.83,0.35,0.09,0.0
3.5,0.92,0.90,0.72,0.32,0.0
5.0,0.11,0.10,0.05,0.00,0.0


R1 held tax in [np.float64(0.03), np.float64(0.035), np.float64(0.04)]  mu_lut in [np.float64(0.01), np.float64(0.03), np.float64(0.06), np.float64(0.1), np.float64(0.15)]  gdiff in [np.float64(0.05), np.float64(0.06), np.float64(0.08)]


**Choice.** `food_inc` is the first control parameter: it is the direct free-energy source (Phi, the dissipation per capita, is its natural partner), and at m_scale 3.5 the R1 extinction fraction crosses from ~0.9 (food_inc 0.008) to ~0.3 (0.013) to 0 (0.018), i.e. the extinction boundary lies inside 0.008-0.018. Fixed other parameters for all example runs: `m_scale=3.5, tax=0.035, mu_lut=0.06, gdiff=0.06` (mid-values of R1's tax/mu_lut/gdiff grids). Init, seeds and egenome follow `run_sim` (GoL LUT, egenome 0b000011, halfplane alive). N=64, 3000 ticks, transient 50%.

## 2026-10-15: first bifurcation diagram (food_inc), example runs

**Question.** Does the post-transient population time series change character (extinct / oscillating / steady / saturated) across the extinction boundary region of `food_inc`?

**What was done.** Eight values of `food_inc`, two seeds, `bif.sweep` with observable `pop`. Everything below is *observed in these example runs, N=64, not a campaign result*.

In [4]:
BASE = dict(m_scale=3.5, tax=0.035, mu_lut=0.06, gdiff=0.06)
VALUES = [0.011, 0.013, 0.015, 0.017, 0.020, 0.024, 0.030, 0.050]
N, NSTEPS = 64, 3000
res = {}
for seed in (0, 1):
    res[seed] = bif.sweep('food_inc', VALUES, base_params=BASE, observable='pop',
                          n_steps=NSTEPS, N=N, seed=seed, transient_frac=0.5)
rows = []
for seed in res:
    for r in res[seed]:
        rows.append(dict(seed=seed, food_inc=r['value'], n_recurrent=r['n_recurrent'],
                         tail_mean=round(r['mean'],1), tail_std=round(r['series_tail_std'],1),
                         min=r['points'].min(), max=r['points'].max()))
tab = pd.DataFrame(rows); display(tab)

,seed,food_inc,n_recurrent,tail_mean,tail_std,min,max
0,0,0.011,1,0.0,0.0,0.000000,0.0
1,0,0.013,16,1294.0,1344.3,11.833333,3924.0
2,0,0.015,20,1594.7,1362.1,99.000000,3941.0
3,0,0.017,40,1937.9,916.4,714.000000,3517.0
4,0,0.020,233,2336.5,90.4,2148.000000,2567.0
5,0,0.024,187,2801.8,47.9,2667.000000,2927.0
6,0,0.030,111,3486.5,26.0,3398.000000,3544.0
7,0,0.050,3,4096.0,0.2,4094.000000,4096.0
8,1,0.011,1,0.0,0.0,0.000000,0.0
9,1,0.013,14,1258.5,1360.6,9.500000,3909.0


In [ ]:
# The simulation may be run interactively with the following
# (commented out so that "Run All" does not block on the SDL window):
# from controls import run_with_controls
# sim = ee.build_sim(dict(BASE, food_inc=0.017), N=128, seed=0)   # any value in VALUES
# run_with_controls(sim, cell_px=4, probes={'ts': True})

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for seed, col in ((0,'C0'), (1,'C3')):
    xs, ys = bif.to_scatter(res[seed])
    ax[0].scatter(xs, ys, s=8, c=col, alpha=.6, label=f'seed {seed}')
ax[0].set_xlabel('food_inc'); ax[0].set_ylabel('population extrema (post-transient)')
ax[0].set_title(f'recurrent extrema of pop, N={N}'); ax[0].legend()
for seed, col in ((0,'C0'), (1,'C3')):
    ax[1].plot([r['value'] for r in res[seed]], [r['series_tail_std'] for r in res[seed]], 'o-', c=col)
ax[1].set_xlabel('food_inc'); ax[1].set_ylabel('tail std of pop'); ax[1].set_title('oscillation amplitude proxy')
plt.tight_layout(); plt.show()

/var/folders/gl/c2ypjywd5zd7__1lddcg09cw0000gs/T/ipykernel_14097/168576117.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Observed (N=64, seeds 0-1).** (i) `food_inc` 0.011: extinct in both seeds (a single point at 0). (ii) 0.013-0.017: large, irregular-amplitude oscillations of the population (tail std ~ 1000-1400 on a mean of ~1300-1800, extrema spanning tens to ~3900 of 4096 sites), so `recurrent_extrema` returns many points (a filled band, not a few discrete branches). (iii) Amplitude shrinks with increasing `food_inc` (0.020 -> 0.030) while the mean rises. (iv) 0.050: pop pinned at ~N^2=4096 (saturation, a trivial fixed point). **No period-doubling-like branch structure is visible**; the diagram is "extinct / large irregular oscillation / shrinking oscillation / saturated fixed point". The reducer counts of local extrema are not by themselves a classification when noise-like fluctuations are present (see ISSUES CRIT-Q2).

### Time series at three representative values

Per-tick `pop` and `Phi` (energy dissipation per capita from `get_dissipation`), with births per tick, and, every 10 ticks, the Bedau-Packard `A_new` (count of real LUT-genome components with cumulative activity above the neutral shadow's p90, computed as in `evoca_explore.run_sim`) and `lut_div`. Values: 0.013 (large oscillation), 0.022 (small oscillation), 0.050 (saturated).

In [6]:
def record(food_inc, N=64, n_steps=3000, seed=0, a_every=10):
    p = dict(BASE, food_inc=food_inc)
    sim = ee.build_sim(p, N=N, seed=seed, init='halfplane')
    sim.neutral_enable()
    out = {k: [] for k in ('pop','phi','births','deaths')}
    out.update(t_a=[], a_new=[], lut_div=[])
    try:
        for t in range(n_steps):
            sim.step()
            d = sim.get_dissipation()
            out['pop'].append(d['alive']); out['phi'].append(d['phi'])
            out['births'].append(sim.get_births_last()); out['deaths'].append(sim.get_deaths_last())
            if t % a_every == 0:
                sim._lib.evoca_activity_update(); sim.n_activity_update()
                G = sim.get_activity(max_n=200000); Nt = sim.get_n_activity(max_n=200000)
                gp = G['pop_count'] > 0; npop = Nt['pop_count'] > 0
                n_acts = Nt['activity'][npop].astype(float)
                th = float(np.percentile(n_acts, 90)) if n_acts.size else 0.0
                out['a_new'].append(int((G['activity'][gp].astype(float) > th).sum()) if gp.any() else 0)
                out['lut_div'].append(int(gp.sum())); out['t_a'].append(t)
    finally:
        pass
    return sim, {k: np.array(v) for k, v in out.items()}

REP = {}
for fi in (0.013, 0.022, 0.050):
    t0 = time.time()
    sim, REP[fi] = record(fi)
    if fi == 0.013:
        sim_keep = sim   # keep for lattice snapshots + recipe
    else:
        sim.free()
    print(fi, 'done in %.1fs' % (time.time()-t0))

0.013 done in 0.6s


0.022 done in 0.8s


0.05 done in 1.2s


In [ ]:
# The simulation may be run interactively with the following
# (commented out so that "Run All" does not block on the SDL window):
# from controls import run_with_controls
# sim = ee.build_sim(dict(BASE, food_inc=0.013), N=128, seed=0)   # boom-bust; also 0.022, 0.050
# run_with_controls(sim, cell_px=4, probes={'ts': True, 'activity': True})

In [7]:
fig, axs = plt.subplots(4, 3, figsize=(14, 9), sharex=True)
for j, fi in enumerate(REP):
    R = REP[fi]; t = np.arange(len(R['pop']))
    axs[0,j].plot(t, R['pop'], lw=.7); axs[0,j].set_title(f'food_inc={fi}')
    axs[1,j].plot(t, R['phi'], lw=.7, c='C1')
    axs[2,j].plot(R['t_a'], R['a_new'], lw=.9, c='C2')
    axs[3,j].plot(R['t_a'], R['lut_div'], lw=.9, c='C4'); axs[3,j].set_xlabel('tick')
for i, l in enumerate(['pop', 'Phi (diss/alive)', 'A_new', 'lut_div']): axs[i,0].set_ylabel(l)
plt.tight_layout(); plt.show()
summ = []
for fi, R in REP.items():
    h = len(R['pop'])//2; x = R['pop'][h:].astype(float) - R['pop'][h:].mean()
    ac = np.correlate(x, x, 'full')[len(x)-1:]; ac = ac/ac[0] if ac[0] > 0 else ac
    pk = [i for i in range(5, len(ac)-1) if ac[i] > ac[i-1] and ac[i] >= ac[i+1] and ac[i] > 0.2]
    summ.append(dict(food_inc=fi, pop_mean=R['pop'][h:].mean(), pop_std=R['pop'][h:].std(),
                     phi_mean=R['phi'][h:].mean(), phi_std=R['phi'][h:].std(),
                     a_new_mean=R['a_new'][len(R['a_new'])//2:].mean(), acf_first_peak_lag=(pk[0] if pk else None)))
display(pd.DataFrame(summ).round(3))

/var/folders/gl/c2ypjywd5zd7__1lddcg09cw0000gs/T/ipykernel_14097/3673452804.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,food_inc,pop_mean,pop_std,phi_mean,phi_std,a_new_mean,acf_first_peak_lag
0,0.013,1293.393,1300.901,0.068,0.118,45.633,126.0
1,0.022,2569.675,59.657,0.035,0.000,412.613,118.0
2,0.050,4095.952,0.226,0.050,0.002,495.993,NaN


**Observed (N=64, seed 0, example runs).** At 0.013 the population cycles with a boom-bust period of ~130 ticks (autocorrelation peaks at multiples of the same lag across seeds 0 and 1 in the preliminary probe), with Phi swinging (phi_std 0.118 on a mean of 0.068) and `A_new` low (mean ~46); at 0.022 a ~118-tick lag peak survives at much smaller amplitude, Phi is pinned at the tax value (0.035, std ~0: all intake is taxed away at steady state) and `A_new` is ~410; at 0.050 pop sits at the lattice ceiling and no autocorrelation peak exists. A_new is much lower in the oscillating regime (repeated population crashes) than in the steady ones, which suggests it, rather than pop extrema, may mark the regime change. Lattice snapshots at a peak and a trough of the 0.013 run:

In [8]:
R = REP[0.013]; sim = sim_keep
fig, axs = plt.subplots(1, 4, figsize=(14, 3.6))
# re-run to a trough and a peak for snapshots
sim_keep.free()
sim = ee.build_sim(dict(BASE, food_inc=0.013), N=64, seed=0)
snaps = {}
want = {int(np.argmax(R['pop'][1500:2000]))+1500: 'peak', int(np.argmin(R['pop'][1500:2000]))+1500: 'trough'}
for t in range(2001):
    if t in want: snaps[want[t]] = (t, rgb(sim, 0), rgb(sim, 2))
    sim.step()
for i, (k, (t, s0, s2)) in enumerate(snaps.items()):
    axs[2*i].imshow(s0); axs[2*i].set_title(f'{k} t={t}: state (mode 0)')
    axs[2*i+1].imshow(s2); axs[2*i+1].set_title(f'{k}: private food (mode 2)')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()
fp = export_recipe_once(sim, 'foodinc013_osc', workstream=WS)   # reuses an existing copy
print('recipe:', fp, '(recipe does not carry the seed; use seed 0 via build_sim for the exact series)')
sim.free()

recipe: /Users/n/Projects/EvoCA/Research/Criticality-bifurcation/Runs/2026-10-15_foodinc013_osc.evoca (recipe does not carry the seed; use seed 0 via build_sim for the exact series)


/var/folders/gl/c2ypjywd5zd7__1lddcg09cw0000gs/T/ipykernel_14097/1163512021.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### N-dependence check (single-point, N = 32, 64, 128)

Is the oscillation a finite-size (synchronized-lattice) effect? Same parameters, `food_inc` in {0.016, 0.022}, seed 0, 3000 ticks. Reported as pop tail mean/std and *relative* amplitude std/mean.

In [9]:
rows = []
for fi in (0.016, 0.022):
    for n in (32, 64, 128):
        s = bif.scalar_series(dict(BASE, food_inc=fi), N=n, n_steps=3000, seed=0)
        tl = s[1500:]; rows.append(dict(food_inc=fi, N=n, mean_density=tl.mean()/n**2, std_density=tl.std()/n**2,
                                        rel_amp=tl.std()/max(tl.mean(),1), min_density=tl.min()/n**2))
nd = pd.DataFrame(rows).round(4); display(nd)

,food_inc,N,mean_density,std_density,rel_amp,min_density
0,0.016,32,0.4312,0.3351,0.7771,0.0605
1,0.016,64,0.4231,0.3183,0.7523,0.0664
2,0.016,128,0.4278,0.2972,0.6948,0.0825
3,0.022,32,0.6273,0.0232,0.0370,0.5586
4,0.022,64,0.6278,0.0210,0.0334,0.5847
5,0.022,128,0.6275,0.0056,0.0090,0.6093


**Observed (single seed, N=32/64/128; not replicated).** At food_inc 0.016 the oscillation amplitude is nearly N-independent in density (std/N^2: 0.335, 0.318, 0.297; relative amplitude 0.78 -> 0.69), i.e. the boom-bust cycle is a *global* synchronized oscillation that does not average out up to N=128 (the slight decline could still be a trend; N=256+ and replicates needed). At 0.022 the relative amplitude falls 0.037 -> 0.033 -> 0.009 from N=64 to 128, which looks like a fluctuation-driven (noise) amplitude that shrinks with size, not a deterministic cycle. So the large-amplitude regime looks intrinsic, the small-amplitude regime looks like finite-size noise. This distinction is the content of ISSUES CRIT-Q1.

### Sanity check against R1

R1 (N=256, 8000 ticks, varying tax / mu_lut / gdiff) recorded extinction for `food_inc` <= 0.010 at m_scale 3.5 and viability by 0.018. Here (N=64, fixed tax 0.035 / mu_lut 0.06 / gdiff 0.06) food_inc 0.011 goes extinct and 0.013-0.050 survive: broadly consistent with the R1 boundary (R1 still had ~32% extinction at 0.013 across its other parameters, so a boundary near 0.012-0.013 at N=64 is plausible). Of R1's rows only one has exactly these tax/mu_lut/gdiff values with m_scale 3.5 (food_inc 0.005, extinct, shown below), so an exact row-level comparison is not possible; the check is at the level of the extinction boundary only. The boundary location is a single-seed N=64 observation (seed 0 at 0.012 was extinct, seed 1 survived in a preliminary probe), i.e. it is stochastic.

In [10]:
sub = r1[(r1.param_m_scale==3.5)&(r1.param_tax==0.035)&(r1.param_mu_lut==0.06)&(r1.param_gdiff==0.06)]
display(sub[['param_food_inc','extinct','alive_density_mean','alive_density_temporal_std','final_pop']].sort_values('param_food_inc'))
print('elapsed so far %.0fs' % (time.time()-T0))

,param_food_inc,extinct,alive_density_mean,alive_density_temporal_std,final_pop
68,0.005,True,0.006183,0.055211,0


elapsed so far 16s


## Current state and directions (2026-10)

- **State.** The harness is merged and now has one honest example: along `food_inc` (m_scale 3.5, tax 0.035, mu_lut 0.06, gdiff 0.06, N=64) the population goes extinct below ~0.012, shows large irregular boom-bust oscillations (~100-130 tick period) between ~0.013 and ~0.017, loses amplitude as food_inc rises, and pins at the lattice ceiling by 0.05. No period-doubling signature is seen; the extrema count is dominated by noise-like amplitude variation. N-dependence is untested beyond a single-seed check.
- **Newly unblocked by the 2026-10-15 merges.** Observables beyond `pop`: Phi (`get_dissipation`), `A_new`, `dyn_activity`, lineage return are all on main, so a diagram can have several y-axes per column. The harness `sweep` records only `pop`/`activity`/`lut_div`, so adding Phi / A_new to `scalar_series` is a dev item (ISSUES CRIT-N2).
- **Decisions and directions.** See the ISSUES snapshot above (CRIT-D1: first control parameter and N; CRIT-D2: what counts as a transition; CRIT-N1..N5).